# 05. Clustering + Soft Membership + Deltas### Objective: K-Means clustering (K=3), compute soft membership, calculate temporal deltas.### Output Artifacts: `../../data/processed/5_clustered_telemetry.csv` (with soft membership + deltas)

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

INPUT_PATH = '../../data/processed/4_activity_contributions.csv'
OUTPUT_PATH = '../../data/processed/5_clustered_telemetry.csv'

print("Libraries imported")

Libraries imported


In [2]:
# Load data
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows")

Loaded 3240 rows


In [3]:
# Prepare features for clustering
features = ['pct_combat', 'pct_collect', 'pct_explore']
X = df[features].fillna(0)

print(f"Features for clustering: {features}")
print(f"Shape: {X.shape}")

Features for clustering: ['pct_combat', 'pct_collect', 'pct_explore']
Shape: (3240, 3)


In [4]:
# K-Means clustering (K=3, PRODUCTION configuration)
print("Running K-Means (K=3)...")
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X)

print("Clustering complete")
print(f"Cluster centers:\n{kmeans.cluster_centers_}")

Running K-Means (K=3)...


Clustering complete
Cluster centers:
[[0.56172902 0.03504491 0.40322607]
 [0.01651481 0.10250127 0.88098392]
 [0.24455772 0.12337346 0.63206882]]


In [5]:
# Map clusters to archetypes (assign each cluster to nearest archetype)
from scipy.spatial.distance import cdist

# Define ideal archetype centers
ideal_combat = np.array([1.0, 0.0, 0.0])  # 100% combat
ideal_collect = np.array([0.0, 1.0, 0.0])  # 100% collection
ideal_explore = np.array([0.0, 0.0, 1.0])  # 100% exploration
ideal_centers = np.array([ideal_combat, ideal_collect, ideal_explore])

# Calculate distances from each cluster to each ideal archetype
cluster_centers = kmeans.cluster_centers_
distances = cdist(cluster_centers, ideal_centers, metric='euclidean')

# Use Hungarian algorithm to assign clusters to archetypes (1-to-1 mapping)
from scipy.optimize import linear_sum_assignment
row_ind, col_ind = linear_sum_assignment(distances)

archetype_names = ['Combat', 'Collection', 'Exploration']
mapping = {row_ind[i]: archetype_names[col_ind[i]] for i in range(len(row_ind))}

df['archetype'] = df['cluster'].map(mapping)
print(f"\n Cluster mapping: {mapping}")
print(f"Archetype distribution:\n{df['archetype'].value_counts()}")


 Cluster mapping: {np.int64(0): 'Combat', np.int64(1): 'Exploration', np.int64(2): 'Collection'}
Archetype distribution:
archetype
Exploration    1867
Collection      970
Combat          403
Name: count, dtype: int64


## Soft Membership Computation (PRODUCTION)Calculate fuzzy membership to each archetype using inverse distance.

In [6]:
# Compute soft membership via inverse distance
print("Computing soft membership...")
distances = kmeans.transform(X)
inv_distances = 1 / (distances + 1e-10)
soft_membership = inv_distances / inv_distances.sum(axis=1, keepdims=True)

# Map to archetype names (now guaranteed to exist)
combat_idx = [k for k, v in mapping.items() if v == 'Combat'][0]
collect_idx = [k for k, v in mapping.items() if v == 'Collection'][0]
explore_idx = [k for k, v in mapping.items() if v == 'Exploration'][0]

df['soft_combat'] = soft_membership[:, combat_idx]
df['soft_collect'] = soft_membership[:, collect_idx]
df['soft_explore'] = soft_membership[:, explore_idx]

print("Soft membership computed")
print(f"Mean soft_combat: {df['soft_combat'].mean():.4f}")
print(f"Mean soft_collect: {df['soft_collect'].mean():.4f}")
print(f"Mean soft_explore: {df['soft_explore'].mean():.4f}")

Computing soft membership...
Soft membership computed
Mean soft_combat: 0.1979
Mean soft_collect: 0.3267
Mean soft_explore: 0.4755


## Delta Computation (PRODUCTION - NEW)Calculate window-to-window changes in soft membership for ANFIS temporal context.

In [7]:
# Compute deltas (per player, sequential)
print("Computing temporal deltas...")
df_sorted = df.sort_values(['userId', 'timestamp'])

df_sorted['delta_combat'] = df_sorted.groupby('userId')['soft_combat'].diff().fillna(0)
df_sorted['delta_collect'] = df_sorted.groupby('userId')['soft_collect'].diff().fillna(0)
df_sorted['delta_explore'] = df_sorted.groupby('userId')['soft_explore'].diff().fillna(0)

df = df_sorted.copy()

print("Deltas computed (first window per player = 0)")
print(f"Δcombat range: [{df['delta_combat'].min():.3f}, {df['delta_combat'].max():.3f}]")
print(f"Δcollect range: [{df['delta_collect'].min():.3f}, {df['delta_collect'].max():.3f}]")
print(f"Δexplore range: [{df['delta_explore'].min():.3f}, {df['delta_explore'].max():.3f}]")

Computing temporal deltas...
Deltas computed (first window per player = 0)
Δcombat range: [-0.836, 0.913]
Δcollect range: [-0.817, 0.830]
Δexplore range: [-0.865, 0.872]


In [8]:
# Save
df.to_csv(OUTPUT_PATH, index=False)
print(f"\n Saved to {OUTPUT_PATH}")
print("Clustering + soft membership + deltas complete!")


 Saved to ../../data/processed/5_clustered_telemetry.csv
Clustering + soft membership + deltas complete!
